# SensiFake v0.1 quickstart

Run from this repository using Python 3.12 and the documented lightweight requirements.
This notebook demonstrates supported dataset access and saved-result verification.
It does not run training, calibration sweeps or model inference.


In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "sensifake_data/published.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))


## Verify saved policy arithmetic (offline)

These are selected policies for the recorded 450-image seed-20 validation run.
They do not establish an independent final-test result.


In [ ]:
import runpy

verifier = runpy.run_path(str(root / "experiments/fusion_calibration/verify_recorded_results.py"))
verifier["verify"]()


## Resolve one image from Hugging Face (network required)

The manifest and image bytes are checked against the pinned revision and SHA-256.
Images remain in memory. This does not construct or change any experimental split.


In [ ]:
import csv
import io
from sensifake_data import PublishedDataset

corpus = PublishedDataset()
rows = list(csv.DictReader(io.StringIO(corpus.manifest_bytes.decode("utf-8"))))
image = corpus.load_image(rows[0])
print(len(rows), image.size, image.mode)


## Optional job-scoped cache

Only verified image bytes enter a new temporary directory, which is disposed when
the context exits. Keep checkpoints, logs and outputs outside this cache.


In [ ]:
with corpus.temporary_cache() as job:
    cached_image = job.load_image(rows[0])
    assert cached_image.size == image.size
